In [3]:
import sys
import pandas as pd

print('Python Version :',sys.version.split()[0])
print('Pandas version :', pd.__version__)

Python Version : 3.13.5
Pandas version : 2.2.3


In [4]:
# to check the current python folder 

from pathlib import Path
print(Path.cwd())

C:\Users\madha


In [7]:
# creating an Audit Record and saving in .json file

from datetime import datetime,timezone
from pathlib import Path
from uuid import uuid4
import json

project_folder = Path(r'C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics')
audit_folder = project_folder/'Documentation'/'ingestion_runs'

audit_folder.mkdir(exist_ok=True)

run_id = str(uuid4())
started_at = datetime.now(timezone.utc).isoformat(timespec='seconds')

run_record = { 'run_id':run_id,
             'started_at':started_at,
             'source_filename':'UK_Gender_Pay_Gap_50000_rows.csv',
             'status': 'started'}
audit_file = audit_folder/f'{run_id}.json'

with audit_file.open("w", encoding="utf-8") as file:
    json.dump(run_record, file, indent=4)

print(f'Audit record is saved in {audit_file}')

Audit record is saved in C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\Documentation\ingestion_runs\9f87ee60-5dbb-4b58-8283-bf0274744ae8.json


In [10]:
raw_pay_gap_df = pd.read_csv(r"C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\raw\UK_Gender_Pay_Gap_50000_rows.csv")
raw_pay_gap_df.head()

,employee_id,gender,department,job_title,job_level,years_at_company,contracted_hours_per_week,work_location,ethnicity,annual_base_pay_gbp,annual_bonus_gbp,total_annual_pay_gbp,employment_type,hire_date
0,E000001,Female,Data,Analyst,4,15.0,25.0,Birmingham,Other,63500,7121.05,70621.05,Part-time,2018-03-18
1,E000002,Male,HR,Analyst,1,13.3,20.0,Bristol,Black,26800,1924.97,28724.97,Fixed-term,2019-07-20
2,E000003,Male,Support,Analyst,3,8.9,40.0,London,White,35200,5334.13,40534.13,Fixed-term,2023-02-18
3,E000004,Female,Operations,Manager,4,2.9,40.0,London,Mixed,51500,2349.34,53849.34,Fixed-term,2019-12-14
4,E000005,Female,Product,Lead,1,2.3,40.0,Leeds,Prefer not to say,35200,2306.93,37506.93,Permanent,2026-02-07


In [13]:
print('Rows and columns: ',raw_pay_gap_df.shape)

Rows and columns:  (50000, 14)


In [17]:
print('Column names :', raw_pay_gap_df.columns.tolist())

Column names : ['employee_id', 'gender', 'department', 'job_title', 'job_level', 'years_at_company', 'contracted_hours_per_week', 'work_location', 'ethnicity', 'annual_base_pay_gbp', 'annual_bonus_gbp', 'total_annual_pay_gbp', 'employment_type', 'hire_date']


In [18]:
raw_pay_gap_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   employee_id                50000 non-null  object 
 1   gender                     50000 non-null  object 
 2   department                 50000 non-null  object 
 3   job_title                  50000 non-null  object 
 4   job_level                  50000 non-null  int64  
 5   years_at_company           50000 non-null  float64
 6   contracted_hours_per_week  50000 non-null  float64
 7   work_location              49750 non-null  object 
 8   ethnicity                  50000 non-null  object 
 9   annual_base_pay_gbp        50000 non-null  int64  
 10  annual_bonus_gbp           50000 non-null  float64
 11  total_annual_pay_gbp       50000 non-null  float64
 12  employment_type            50000 non-null  object 
 13  hire_date                  50000 non-null  obj

In [12]:
raw_pay_gap_df.describe()

,job_level,years_at_company,contracted_hours_per_week,annual_base_pay_gbp,annual_bonus_gbp,total_annual_pay_gbp
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,2.441920,11.007536,31.304350,41879.356000,3562.157440,45539.499440
std,1.179175,6.358011,7.047697,16026.337782,2582.500222,17396.549623
min,1.000000,0.000000,20.000000,-100.000000,0.210000,18001.870000
25%,2.000000,5.500000,25.000000,30100.000000,1561.615000,32722.195000
50%,2.000000,11.000000,35.000000,38600.000000,3126.990000,41901.125000
75%,3.000000,16.500000,37.500000,50200.000000,4980.677500,54642.677500
max,5.000000,22.000000,40.000000,127700.000000,19967.940000,145336.460000


#### output showed job levels 1–5, tenure 0–22 years, contracted hours 20–40 per week, and a minimum annual base pay of −£100.

In [19]:
# updating the Audit Record with successful status for loading the data and for investigating the structure

actual_row_count, actual_column_count = raw_pay_gap_df.shape

run_record.update({'csv_load_status':'succeeded',
                  'actual_row_count':actual_row_count,
                  'actual_column_count': actual_column_count})

with audit_file.open("w", encoding="utf-8") as file:
    json.dump(run_record, file, indent=4)

print(f'Audit record updated successfully:{audit_file.name}')
print(f'CSV loaded succesfully: with {actual_row_count} rows and {actual_column_count} columns')

Audit record updated successfully:9f87ee60-5dbb-4b58-8283-bf0274744ae8.json
CSV loaded succesfully: with 50000rows and 14 columns


In [28]:
# Starting Data Profiling 

missing_count = raw_pay_gap_df.isna().sum()
blank_count = raw_pay_gap_df.apply(lambda columns: columns.astype('string').str.strip().eq('').sum())

missing_blank_summary_df = pd.DataFrame({'missing_count':missing_count,
                           'blank_count':blank_count})
display(missing_blank_summary_df)

,missing_count,blank_count
employee_id,0,0
gender,0,0
department,0,0
job_title,0,0
job_level,0,0
years_at_company,0,0
contracted_hours_per_week,0,0
work_location,250,0
ethnicity,0,0
annual_base_pay_gbp,0,0


In [33]:
gender_count = raw_pay_gap_df['gender'].value_counts(dropna=False)

for value,count in gender_count.items():
    print(f'{value!r}:', count)

'Female': 24248
'Male': 24119
'Prefer not to say': 873
'Non-binary': 600
'unknown': 160


In [61]:
department_count= raw_pay_gap_df['department'].value_counts(dropna=False)

for value,count in department_count.items():
    print(f'{value!r}:',count)

'Product': 5093
'Marketing': 5036
'Engineering': 5020
'Data': 5001
'Support': 4997
'Legal': 4992
'Finance': 4986
'HR': 4971
'Sales': 4955
'Operations': 4949


In [62]:
job_title_count= raw_pay_gap_df['job_title'].value_counts(dropna=False)

for value,count in job_title_count.items():
    print(f'{value!r}:',count)

'Lead': 10136
'Manager': 10105
'Analyst': 9991
'Senior Analyst': 9898
'Specialist': 9870


In [63]:
work_location_count= raw_pay_gap_df['work_location'].value_counts(dropna=False)

for value,count in work_location_count.items():
    print(f'{value!r}:',count)

'Leeds': 7183
'Glasgow': 7128
'Bristol': 7100
'Manchester': 7097
'Birmingham': 7083
'London': 7082
'Remote': 7077
nan: 250


In [64]:
ethnicity_count= raw_pay_gap_df['ethnicity'].value_counts(dropna=False)

for value,count in ethnicity_count.items():
    print(f'{value!r}:',count)

'Asian': 8418
'Prefer not to say': 8380
'Black': 8312
'White': 8310
'Mixed': 8305
'Other': 8275


In [65]:
employment_type_count= raw_pay_gap_df['employment_type'].value_counts(dropna=False)

for value,count in employment_type_count.items():
    print(f'{value!r}:',count)

'Fixed-term': 16798
'Part-time': 16631
'Permanent': 16571


#### Observation -- gender contains five distinct values. The label 'unknown' appears in 160 rows (0.32%), and 'Prefer not to say' appears in 873 rows. No nulls or blank strings were detected.

In [36]:
# identifying null or missing values on employee_id

employee_ids = raw_pay_gap_df['employee_id'].astype('string').str.strip()

missing_key = employee_ids.isna() | employee_ids.eq('')
duplicated_key = ~missing_key & employee_ids.duplicated(keep=False)

print('Rows with the missing or blank employee_ids:',missing_key.sum())
print('Rows with repeated employee_ids:',duplicated_key.sum())

Rows with the missing or blank employee_ids: 0
Rows with repeated employee_ids: 0


In [42]:
# Checking for negative base pay 

negative_base_pay = raw_pay_gap_df['annual_base_pay_gbp']<0

print('Rows with neagtive annual base pay:',negative_base_pay.sum())


Rows with neagtive annual base pay: 120


In [43]:
negative_base_pay_rows_df = raw_pay_gap_df.loc[negative_base_pay].copy()
negative_base_pay_rows_df.head()

,employee_id,gender,department,job_title,job_level,years_at_company,contracted_hours_per_week,work_location,ethnicity,annual_base_pay_gbp,annual_bonus_gbp,total_annual_pay_gbp,employment_type,hire_date
153,E000154,Male,Legal,Specialist,1,13.1,30.0,Manchester,Asian,-100,1543.25,32343.25,Fixed-term,2022-04-28
414,E000415,Female,Product,Manager,3,3.0,25.0,Birmingham,Mixed,-100,10204.11,75004.11,Fixed-term,2022-05-20
2690,E002691,Male,Product,Specialist,2,6.1,37.5,Manchester,White,-100,1799.24,47299.24,Part-time,2024-03-28
2962,E002963,Female,Operations,Analyst,4,15.6,25.0,Glasgow,Mixed,-100,4899.16,43399.16,Permanent,2023-04-27
3556,E003557,Female,Finance,Manager,2,6.1,20.0,London,Other,-100,5914.14,42514.14,Permanent,2022-04-09


In [44]:
# checking hire_date and parse into DATETIME

display(raw_pay_gap_df['hire_date'].head())

0    2018-03-18
1    2019-07-20
2    2023-02-18
3    2019-12-14
4    2026-02-07
Name: hire_date, dtype: object

In [46]:
hire_date_text = raw_pay_gap_df['hire_date'].astype('string').str.strip()
hire_date_parsed = pd.to_datetime(hire_date_text, format='%Y-%m-%d', errors='coerce')

invalid_date_mask = hire_date_text.notna()&hire_date_text.ne('')&hire_date_parsed.isna()

print('Earliest parsed hire date:',hire_date_parsed.min())
print('Latest parsed hire date:',hire_date_parsed.max())
print('Rows with unparseable hire date:',invalid_date_mask.sum())


Earliest parsed hire date: 2018-01-01 00:00:00
Latest parsed hire date: 2026-10-05 00:00:00
Rows with unparseable hire date: 0


In [49]:
print(raw_pay_gap_df['hire_date'].dtype)

object


In [52]:
raw_pay_gap_df['hire_date']= raw_pay_gap_df['hire_date'].astype('string').str.strip()

raw_pay_gap_df['hire_date']=pd.to_datetime(raw_pay_gap_df['hire_date'], format='%Y-%m-%d', errors='coerce')
print(raw_pay_gap_df['hire_date'].dtype)

datetime64[ns]


In [55]:
raw_pay_gap_df['hire_date'].isna().sum()

np.int64(0)

In [53]:
print(raw_pay_gap_df['job_level'].dtype)

int64


In [56]:
print(raw_pay_gap_df['job_level'].isna().sum())
print(raw_pay_gap_df['job_level'].min())
print(raw_pay_gap_df['job_level'].max())

0
1
5


In [54]:
print(raw_pay_gap_df['years_at_company'].dtype)

float64


In [57]:
print(raw_pay_gap_df['contracted_hours_per_week'].isna().sum())
print(raw_pay_gap_df['contracted_hours_per_week'].min())
print(raw_pay_gap_df['contracted_hours_per_week'].max())

0
20.0
40.0


In [58]:
print(raw_pay_gap_df['years_at_company'].isna().sum())
print(raw_pay_gap_df['years_at_company'].min())
print(raw_pay_gap_df['years_at_company'].max())

0
0.0
22.0


In [60]:
print(raw_pay_gap_df['years_at_company'].dtype)

float64


In [66]:
raw_pay_gap_df["annual_base_pay_gbp"] = raw_pay_gap_df["annual_base_pay_gbp"].round(2)
raw_pay_gap_df["annual_bonus_gbp"] = raw_pay_gap_df["annual_bonus_gbp"].round(2)
raw_pay_gap_df["total_annual_pay_gbp"] = raw_pay_gap_df["total_annual_pay_gbp"].round(2)

In [68]:
# logging the conversion failure 

source_file = project_folder / "raw" / "UK_Gender_Pay_Gap_50000_rows.csv"
source_check_df = pd.read_csv(source_file)

pay_columns = [
    "annual_base_pay_gbp",
    "annual_bonus_gbp",
    "total_annual_pay_gbp",
]

conversion_failures = []

for column in ["hire_date", *pay_columns]:
    original = source_check_df[column]
    original_text = original.astype("string").str.strip()
    populated = original_text.notna() & original_text.ne("").fillna(False)
    actual = raw_pay_gap_df[column]

    if column == "hire_date":
        expected_format = original_text.str.fullmatch(
            r"[0-9]{4}-[0-9]{2}-[0-9]{2}", na=False
        )
        expected = pd.to_datetime(
            original_text, format="%Y-%m-%d", errors="coerce"
        )
        invalid_source = ~expected_format | expected.isna()
        type_ok = pd.api.types.is_datetime64_any_dtype(actual)
        reason = "Expected a valid YYYY-MM-DD date matching the converted value"
    else:
        numeric = pd.to_numeric(original_text, errors="coerce")
        invalid_source = numeric.isna() | numeric.isin(
            [float("inf"), float("-inf")]
        )
        expected = numeric.round(2)
        type_ok = pd.api.types.is_numeric_dtype(actual)
        reason = "Expected a finite numeric value rounded to two decimal places"

    if type_ok:
        failed = populated & (invalid_source | ~actual.eq(expected).fillna(False))
    else:
        failed = populated
        reason = f"Unexpected output dtype: {actual.dtype}"

    for row_index in source_check_df.index[failed]:
        conversion_failures.append({
            "employee_id": source_check_df.at[row_index, "employee_id"],
            "column": column,
            "original_value": original.at[row_index],
            "result": actual.at[row_index],
            "reason": reason,
        })

conversion_log_df = pd.DataFrame(
    conversion_failures,
    columns=["employee_id", "column", "original_value", "result", "reason"],
)

print("Failed checks:", len(conversion_log_df))
display(conversion_log_df)

Failed checks: 0


,employee_id,column,original_value,result,reason


In [69]:
# saving the result into existing audit JSON

failure_count = len(conversion_log_df)

run_record["conversion_failure_count"] = failure_count
run_record["conversion_check_status"] = "passed" if failure_count == 0 else "failed"

if failure_count > 0:
    conversion_log_file = audit_file.with_name(
        f"{run_record['run_id']}_conversion_failures.csv"
    )
    conversion_log_df.to_csv(conversion_log_file, index=False)
    run_record["conversion_log_file"] = conversion_log_file.name
else:
    run_record.pop("conversion_log_file", None)

with audit_file.open("w", encoding="utf-8") as file:
    json.dump(run_record, file, indent=4)

print(f"Audit record updated: {audit_file.name}")

Audit record updated: 9f87ee60-5dbb-4b58-8283-bf0274744ae8.json


In [70]:
print(audit_file.resolve())
print(audit_file.read_text(encoding="utf-8"))

C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\Documentation\ingestion_runs\9f87ee60-5dbb-4b58-8283-bf0274744ae8.json
{
    "run_id": "9f87ee60-5dbb-4b58-8283-bf0274744ae8",
    "started_at": "2026-11-19T13:36:20+00:00",
    "source_filename": "UK_Gender_Pay_Gap_50000_rows.csv",
    "status": "started",
    "csv_load_status": "succeeded",
    "actual_row_count": 50000,
    "actual_column_count": 14,
    "conversion_failure_count": 0,
    "conversion_check_status": "passed"
}


In [71]:
# Closing the existing Audit Run

from datetime import datetime, timezone

run_record["status"] = "succeeded"
run_record["finished_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")

with audit_file.open("w", encoding="utf-8") as file:
    json.dump(run_record, file, indent=4)

print(audit_file.read_text(encoding="utf-8"))

{
    "run_id": "9f87ee60-5dbb-4b58-8283-bf0274744ae8",
    "started_at": "2026-11-19T13:36:20+00:00",
    "source_filename": "UK_Gender_Pay_Gap_50000_rows.csv",
    "status": "succeeded",
    "csv_load_status": "succeeded",
    "actual_row_count": 50000,
    "actual_column_count": 14,
    "conversion_failure_count": 0,
    "conversion_check_status": "passed",
    "finished_at": "2026-11-20T00:19:11+00:00"
}
